# BCR Stage 2 — Model Ablation (Colab GPU)

Run this notebook on a **Colab A100 or T4 runtime** (`Runtime > Change runtime type > GPU`).

Stage 2 trains three models (XGBoost, LightGBM, CatBoost) using the label
strategy selected in Stage 1 as a fixed input. GPU acceleration is enabled
automatically via `get_device()`.

**Expected runtime:** ~8 min on T4, ~4 min on A100.

Results are written to `results/stage2_results.json` and
`results/stage2_report.md`, and logged to Weights & Biases.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# Install dependencies and mount Google Drive.

!pip install -q lightgbm catboost xgboost wandb python-dotenv scikit-learn pandas numpy

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Repo & data ───────────────────────────────────────────────────────
# Clone the repo (or adjust the path if you uploaded it to Drive directly)
# and copy raw data from Drive into the expected relative path.

import os, sys, shutil
from pathlib import Path

REPO_DIR = Path('/content/eeg_pipeline')
DRIVE_DATA = Path('/content/drive/MyDrive/eeg_pipeline/data/raw/Bad_channels_for_ML.csv')

if not REPO_DIR.exists():
    !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/eeg_pipeline

# Make the bad_channel_rejection package importable
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

# Copy raw CSV from Drive into the path the module expects
dest = REPO_DIR / 'data' / 'raw'
dest.mkdir(parents=True, exist_ok=True)
shutil.copy(DRIVE_DATA, dest / 'Bad_channels_for_ML.csv')
print('Data ready:', (dest / 'Bad_channels_for_ML.csv').stat().st_size, 'bytes')

In [ ]:
# ── Cell 3: W&B auth ──────────────────────────────────────────────────────────
# Override WANDB_MODE to online so results are synced to your W&B workspace.
# Paste your API key from https://wandb.ai/authorize when prompted.

import os
os.environ['WANDB_MODE'] = 'online'   # override the offline default from .env

import wandb
wandb.login()   # paste API key when prompted

In [ ]:
# ── Cell 4: Run Stage 2 ───────────────────────────────────────────────────────
# Change --winning-strategy to match your Stage 1 winner.
# Add --model lightgbm to run only one model instead of all three.

!python -m bad_channel_rejection.ablation_stage2_models \
    --winning-strategy dawid_skene

In [ ]:
# ── Cell 5: Results ───────────────────────────────────────────────────────────

import json
from pathlib import Path

results_path = Path('results/stage2_results.json')
stage2 = json.loads(results_path.read_text())

print(f"Label strategy: {stage2['label_strategy']}")
print()
print(f"{'Model':<12} {'AUPRC mean':>12} {'AUPRC std':>10}")
print('-' * 36)
for model, r in stage2['conditions'].items():
    marker = '  <-- winner' if model == stage2['winner_model'] else ''
    print(f"{model:<12} {r['auprc_mean']:>12.4f} {r['auprc_std']:>10.4f}{marker}")

print()
print(f"Winner: {stage2['winner_model']}")
print()
print(Path('results/stage2_report.md').read_text())